In [18]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from gamePredModel import GamePredModel, mdn_loss, players_and_stats
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

In [19]:
INPUT_DIM = 420
HIDDEN_DIM = 4096
OUTPUT_DIM = 180
N_COMPONENTS = 2
BATCH_SIZE = 32
GAME_ID = 22400367

In [ ]:




model = GamePredModel(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, output_dim=OUTPUT_DIM, n_components=N_COMPONENTS)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = mdn_loss
num_epochs = 3

df = pd.read_csv("../csv/masterGame.csv")

# Step 2: Split features and target
X = df.iloc[:, -420:]
y = df.iloc[:, 6:-420]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# # Step 4: To tensors
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32)

# Step 5: DataLoader
train_loader = DataLoader(
    TensorDataset(X_train_tensor, y_train_tensor), batch_size=BATCH_SIZE, shuffle=True
)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=BATCH_SIZE)
print(df.shape)
df.describe()

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    epoch_start = time.time()

    for i, (batch_x, batch_y) in enumerate(train_loader, 1):
        batch_start = time.time()
        optimizer.zero_grad()

        # Forward pass
        pi, mu, sigma = model(batch_x)

        # Loss computation
        loss = mdn_loss(pi, mu, sigma, batch_y)
        total_loss += loss.item()

        # Backpropagation
        loss.backward()
        optimizer.step()

        batch_time = time.time() - batch_start
        if i % 100 == 0 or i == len(train_loader):
            print(
                f"Batch {i:3d}/{len(train_loader)} - Loss: {loss.item():.6f} - Time: {batch_time:.2f}s"
            )

    avg_train_loss = total_loss / len(train_loader)
    epoch_time = time.time() - epoch_start

    # ----- Evaluation -----
    model.eval()
    total_test_loss = 0.0
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            pi, mu, sigma = model(batch_x)
            test_loss = mdn_loss(pi, mu, sigma, batch_y)
            total_test_loss += test_loss.item()

    avg_test_loss = total_test_loss / len(test_loader)

    # ----- Logging -----
    print(f"\nEpoch [{epoch+1}/{num_epochs}] Summary:")
    print(f"  Train Loss : {avg_train_loss:.6f}")
    print(f"  Test Loss  : {avg_test_loss:.6f}")
    print(f"  Epoch Time : {epoch_time:.2f}s")
    print("-" * 60)


(50646, 606)
Batch 100/1267 - Loss: 1.576847 - Time: 0.04s
Batch 200/1267 - Loss: 1.350944 - Time: 0.05s
Batch 300/1267 - Loss: 1.139496 - Time: 0.05s
Batch 400/1267 - Loss: 0.866256 - Time: 0.04s
Batch 500/1267 - Loss: 0.659202 - Time: 0.05s
Batch 600/1267 - Loss: 0.402075 - Time: 0.05s
Batch 700/1267 - Loss: 0.114194 - Time: 0.05s
Batch 800/1267 - Loss: -0.035770 - Time: 0.04s
Batch 900/1267 - Loss: -0.086494 - Time: 0.04s
Batch 1000/1267 - Loss: -0.119785 - Time: 0.04s
Batch 1100/1267 - Loss: -0.255725 - Time: 0.04s
Batch 1200/1267 - Loss: -0.385205 - Time: 0.04s
Batch 1267/1267 - Loss: -0.184323 - Time: 0.05s

Epoch [1/3] Summary:
  Train Loss : 0.532427
  Test Loss  : -0.324111
  Epoch Time : 61.31s
------------------------------------------------------------
Batch 100/1267 - Loss: -0.167822 - Time: 0.04s
Batch 200/1267 - Loss: -0.195944 - Time: 0.04s
Batch 300/1267 - Loss: -0.035185 - Time: 0.04s
Batch 400/1267 - Loss: 0.104692 - Time: 0.05s
Batch 500/1267 - Loss: -0.153475 - Tim

In [35]:
# # After training
torch.save(model.state_dict(), "game_pred_mdn.pt")

In [36]:
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
}, "game_pred_mdn_full.pt")

In [37]:
model = GamePredModel(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, output_dim=OUTPUT_DIM, n_components=N_COMPONENTS)
model.load_state_dict(torch.load("game_pred_mdn.pt"))
model.eval()  # set to evaluation mode

checkpoint = torch.load("game_pred_mdn_full.pt")
model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

In [38]:
data = df[df["gameId"] == GAME_ID]
X_input = data.iloc[:, -420:]
data

,gameId,homeScore,awayScore,encodedHomeTeam,encodedAwayTeam,season,t0_p0_points,t0_p0_assists,t0_p0_blocks,t0_p0_steals,...,t1_p5_numMinutes,t1_p6_numMinutes,t1_p7_numMinutes,t1_p8_numMinutes,t1_p9_numMinutes,t1_p10_numMinutes,t1_p11_numMinutes,t1_p12_numMinutes,t1_p13_numMinutes,t1_p14_numMinutes
825,22400367,107,133,18,21,2024,17.0,7.0,1.0,3.0,...,27.48,18.46,18.15,5.05,4.06,1.5,0.0,0.0,0.0,0.0


In [39]:
X_tensor = torch.tensor(X_input.values, dtype=torch.float32)
X_tensor = X_tensor.view(1, -1)
print(X_tensor)
model.eval()  # if not already in eval mode
with torch.no_grad():
    pi, mu, sigma = model(X_tensor)


tensor([[3.6175e+01, 2.7212e+01, 4.4875e+00, 6.3750e-01, 1.1375e+00, 2.0150e+01,
         9.0125e+00, 1.0137e+01, 4.0000e+00, 6.2000e+00, 5.1875e+00, 5.6250e+00,
         3.1125e+00, 3.2061e+01, 1.8667e+01, 4.6812e+00, 2.3188e-01, 6.8116e-01,
         1.3609e+01, 6.5942e+00, 4.5507e+00, 1.5652e+00, 4.8551e+00, 3.9130e+00,
         7.0580e+00, 2.8406e+00, 2.5720e+01, 1.1476e+01, 3.5714e+00, 2.6984e-01,
         1.1587e+00, 9.4127e+00, 3.9683e+00, 6.9524e+00, 2.7619e+00, 1.0000e+00,
         7.7778e-01, 3.6190e+00, 1.5714e+00, 3.1675e+01, 1.2195e+01, 1.9878e+00,
         9.0244e-01, 1.3415e+00, 1.0171e+01, 4.8537e+00, 3.6951e+00, 1.2195e+00,
         1.5610e+00, 1.2683e+00, 5.7317e+00, 1.1585e+00, 3.2946e+01, 1.2028e+01,
         1.7639e+00, 1.4444e+00, 7.7778e-01, 7.0833e+00, 4.7361e+00, 0.0000e+00,
         0.0000e+00, 3.7917e+00, 2.5556e+00, 1.0903e+01, 1.2361e+00, 2.5079e+01,
         9.4268e+00, 2.7195e+00, 4.1463e-01, 6.0976e-01, 7.5122e+00, 3.2927e+00,
         4.5122e+00, 1.7195e

In [40]:
pi

tensor([[[7.1763e-06, 9.9999e-01],
         [3.9613e-01, 6.0387e-01],
         [5.6054e-01, 4.3946e-01],
         [4.5878e-01, 5.4122e-01],
         [4.5634e-01, 5.4366e-01],
         [3.2686e-01, 6.7314e-01],
         [9.9999e-01, 5.3950e-06],
         [3.7926e-01, 6.2074e-01],
         [4.2244e-01, 5.7756e-01],
         [5.2146e-01, 4.7854e-01],
         [6.0189e-01, 3.9811e-01],
         [6.3927e-01, 3.6073e-01],
         [2.6846e-04, 9.9973e-01],
         [6.4662e-01, 3.5338e-01],
         [9.9637e-01, 3.6296e-03],
         [4.0894e-01, 5.9106e-01],
         [3.8474e-01, 6.1526e-01],
         [8.9199e-01, 1.0801e-01],
         [3.0032e-03, 9.9700e-01],
         [3.0260e-01, 6.9740e-01],
         [3.4210e-01, 6.5790e-01],
         [4.2021e-01, 5.7979e-01],
         [4.6562e-01, 5.3438e-01],
         [3.3728e-01, 6.6272e-01],
         [9.9437e-01, 5.6279e-03],
         [3.9407e-01, 6.0593e-01],
         [3.6751e-01, 6.3249e-01],
         [6.2106e-01, 3.7894e-01],
         [5.8514e-01

In [41]:
mu

tensor([[[ 9.3392e-01,  2.4105e+01],
         [ 9.2901e+00,  3.4338e+00],
         [-3.4583e-03,  2.1413e+00],
         [ 2.6972e+00,  5.6770e-01],
         [ 5.0958e+00,  1.1326e+01],
         [ 4.7558e+00,  1.9777e+00],
         [ 2.1205e+01,  9.6345e-02],
         [ 8.5239e+00,  2.9930e+00],
         [ 2.0905e+00, -1.1427e-02],
         [ 5.2645e-01,  2.6619e+00],
         [ 4.9966e+00,  1.1240e+01],
         [ 1.5770e+00,  4.2139e+00],
         [-3.7592e-02,  1.8001e+01],
         [ 2.2775e+00,  7.3075e+00],
         [ 8.2927e-01, -2.4498e-01],
         [ 2.4286e+00,  5.3953e-01],
         [ 1.0561e+01,  4.5402e+00],
         [ 2.4279e+00,  1.2399e+00],
         [ 2.6075e+00,  1.4708e+01],
         [ 6.3290e+00,  1.9878e+00],
         [ 2.8517e-03,  1.5308e+00],
         [-6.2398e-03,  1.9299e+00],
         [ 9.4819e+00,  3.8662e+00],
         [ 3.3069e+00,  1.0229e+00],
         [ 1.2020e+01,  2.4735e+00],
         [ 5.0599e+00,  1.4134e+00],
         [ 1.8892e+00,  1.2424e-03],
 

In [42]:
sigma

tensor([[[9.9300e-01, 9.7993e+00],
         [4.6272e+00, 1.9293e+00],
         [1.7291e-03, 1.4370e+00],
         [1.4585e+00, 5.1846e-01],
         [2.1779e+00, 5.4048e+00],
         [2.1973e+00, 1.2611e+00],
         [9.4364e+00, 3.1460e-01],
         [4.3531e+00, 1.8724e+00],
         [1.3410e+00, 2.7173e-03],
         [5.0848e-01, 1.4367e+00],
         [2.4180e+00, 5.1049e+00],
         [1.0059e+00, 1.9696e+00],
         [2.4034e-02, 8.4651e+00],
         [1.4736e+00, 3.8498e+00],
         [1.2010e+00, 4.9053e-03],
         [1.3284e+00, 4.8207e-01],
         [4.6913e+00, 2.3178e+00],
         [1.9447e+00, 5.7535e-02],
         [1.0366e-01, 7.6891e+00],
         [3.4850e+00, 1.3618e+00],
         [3.1701e-03, 1.3053e+00],
         [2.4135e-03, 9.8897e-01],
         [4.4101e+00, 2.1236e+00],
         [1.7788e+00, 8.0253e-01],
         [6.7016e+00, 1.4849e-01],
         [2.8788e+00, 9.7753e-01],
         [1.1968e+00, 1.7753e-03],
         [1.0416e+00, 1.7552e-03],
         [1.7545e+00

In [43]:
def sample_from_mdn(pi, mu, sigma):
    """
    pi:    (B, D, K) — mixture weights
    mu:    (B, D, K) — means
    sigma: (B, D, K) — stds

    Returns:
        samples: (B, D)
    """
    B, D, K = pi.shape

    # Step 1: Sample a component index from categorical distribution for each output dim
    categorical = torch.distributions.Categorical(pi)
    component_indices = categorical.sample()  # (B, D) — mixture component chosen per output

    # Step 2: Gather mu and sigma corresponding to sampled component
    batch_indices = torch.arange(B).unsqueeze(1).expand(B, D)  # (B, D)
    dim_indices = torch.arange(D).unsqueeze(0).expand(B, D)    # (B, D)

    # Gather the corresponding mu and sigma based on sampled component
    chosen_mu = mu[batch_indices, dim_indices, component_indices]
    chosen_sigma = sigma[batch_indices, dim_indices, component_indices]

    # Step 3: Sample from normal distribution
    normal = torch.distributions.Normal(chosen_mu, chosen_sigma)
    samples = normal.sample()  # (B, D)
    samples = torch.clamp(samples, min=0.0)

    return samples

In [199]:
samples = sample_from_mdn(pi, mu, sigma)  # Shape: (1, 360)

In [200]:
h, a = players_and_stats(samples, GAME_ID, "../csv/masterGame.csv", "../csv/PlayerStatistics.csv")

Schema({'firstName': String, 'lastName': String, 'personId': Int64, 'gameId': Int64, 'gameDate': String, 'playerteamCity': String, 'playerteamName': String, 'opponentteamCity': String, 'opponentteamName': String, 'gameType': String, 'gameLabel': String, 'gameSubLabel': String, 'seriesGameNumber': Float64, 'win': Int64, 'home': Int64, 'numMinutes': Float64, 'points': Float64, 'assists': Float64, 'blocks': Float64, 'steals': Float64, 'fieldGoalsAttempted': Float64, 'fieldGoalsMade': Float64, 'fieldGoalsPercentage': Float64, 'threePointersAttempted': Float64, 'threePointersMade': Float64, 'threePointersPercentage': Float64, 'freeThrowsAttempted': Float64, 'freeThrowsMade': Float64, 'freeThrowsPercentage': Float64, 'reboundsDefensive': Float64, 'reboundsOffensive': Float64, 'reboundsTotal': Float64, 'foulsPersonal': Float64, 'turnovers': Float64, 'plusMinusPoints': Float64, 'encodedTeam': Int64})
18 21


In [201]:
print(h["points"].sum(), a["points"].sum())

117.1013412475586 140.3069610595703


In [202]:
h

firstName,lastName,personId,points,assists,blocks,steals,reboundsTotal,turnovers
str,str,i64,f32,f32,f32,f32,f32,f32
"""Anthony""","""Edwards""",1630162,25.803543,9.661681,0.0,0.947284,9.076491,1.751937
"""Julius""","""Randle""",203944,39.267982,3.921318,0.224781,0.830468,9.787584,0.767914
"""Donte""","""DiVincenzo""",1628978,10.460669,2.587386,0.0,0.792567,2.908532,0.939958
"""Jaden""","""McDaniels""",1630183,6.931751,0.0,0.010324,0.0,11.6703,2.813545
"""Rudy""","""Gobert""",203497,11.035434,1.689722,0.003017,2.051392,4.685094,4.405427
…,…,…,…,…,…,…,…,…
"""Luka""","""Garza""",1630568,0.0,0.009612,0.0,0.0,0.014725,0.0
"""Josh""","""Minott""",1631169,0.0,0.0,0.0,0.007492,0.0,0.0
"""PJ""","""Dozier""",1628408,0.003239,0.019572,0.0,0.0,0.0,0.004313


In [203]:
a

firstName,lastName,personId,points,assists,blocks,steals,reboundsTotal,turnovers
str,str,i64,f32,f32,f32,f32,f32,f32
"""Karl-Anthony""","""Towns""",1626157,24.767124,13.377798,0.02243,0.0,3.038804,0.856735
"""Mikal""","""Bridges""",1628969,14.453997,5.02913,2.606043,0.728308,10.440793,9.481155
"""OG""","""Anunoby""",1628384,27.45118,12.225641,0.00204,2.314866,10.908648,1.49482
"""Jalen""","""Brunson""",1628973,24.209377,1.143102,1.022304,0.003147,8.669761,1.406437
"""Miles""","""McBride""",1630540,8.667492,11.1581,0.001319,0.577933,14.273738,1.795496
…,…,…,…,…,…,…,…,…
"""Ariel""","""Hukporti""",1630574,0.0,0.005204,0.0,0.013652,0.003178,0.0
"""Tyler""","""Kolek""",1642278,0.000803,0.0,0.0,0.001046,0.0,0.0
"""Jacob""","""Toppin""",1631210,0.003803,0.0,0.003229,0.0,0.00247,0.008193
